# Notebook 04 — Clinical Decision Support Training

## Goal

Train a complete multimodal clinical decision support model that:

1. Accepts image features (256-d), text embeddings (768-d), and clinical features
2. Passes them through a **FusionEncoder** → shared latent representation (256-d)
3. Feeds the representation + clinical features into a **DecisionHead** → Clinical Risk (Low / Medium / High)
4. Saves the unified representation alongside the trained model for downstream retrieval / clustering / explainability


In [1]:
from pathlib import Path

import json
import numpy as np
import pandas as pd
import joblib

import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report
from tqdm.auto import tqdm


## 1. Configuration

In [2]:
#TEXT_ENCODER = "clinicalbert"
TEXT_ENCODER = "biobert"

FUSION_DIR = Path(
    f"/kaggle/input/datasets/mariammohamed1095/models/datasets/processed/fusion/{TEXT_ENCODER}"
)

CLINICAL_DIR = Path(
    "/kaggle/input/datasets/mariammohamed1095/modelsss/datasets/processed/clinical_features"
)

MODEL_DIR = Path("/kaggle/working/models/fusion")
MODEL_DIR.mkdir(parents=True, exist_ok=True)

REPR_DIR = Path("/kaggle/working/reports/fusion/representations")
REPR_DIR.mkdir(parents=True, exist_ok=True)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)


Device: cuda


## 2. Load Data

In [3]:
def load_split(split):
    fusion   = np.load(FUSION_DIR / f"{split}_fusion.npz")
    clinical = pd.read_csv(CLINICAL_DIR / f"{split}_clinical_features.csv")
    return {
        "image":    fusion["image_features"],
        "text":     fusion["text_features"],
        "clinical": clinical,
    }

train      = load_split("train")
validation = load_split("validation")
test       = load_split("test")

for name, data in {"Train": train, "Validation": validation, "Test": test}.items():
    print("=" * 55)
    print(name)
    print("  image   :", data["image"].shape)
    print("  text    :", data["text"].shape)
    print("  clinical:", data["clinical"].shape)


Train
  image   : (257, 256)
  text    : (257, 768)
  clinical: (257, 14)
Validation
  image   : (56, 256)
  text    : (56, 768)
  clinical: (56, 14)
Test
  image   : (56, 256)
  text    : (56, 768)
  clinical: (56, 14)


## 3. Rule-Based Label Generation

Scores are computed from TRAIN statistics only (thresholds fit on train).
Validation and test use the same thresholds — no leakage.

Scoring rubric (from design spec):

| Feature               | Score |
|-----------------------|-------|
| WT Volume large       | +2    |
| ET Volume large       | +2    |
| TC Volume large       | +1    |
| Edema present         | +1    |
| Necrosis present      | +2    |
| Ventricular compress  | +2    |
| Multi-lobe (≥3)       | +1    |
| Bilateral             | +1    |

**Risk Levels:** 0–2 → Low (0) · 3–5 → Medium (1) · 6+ → High (2)


In [4]:
# Thresholds computed from TRAIN ONLY
WT_THRESHOLD = train["clinical"]["wt_volume"].quantile(0.75)
TC_THRESHOLD = train["clinical"]["tc_volume"].quantile(0.75)
ET_THRESHOLD = train["clinical"]["et_volume"].quantile(0.75)

print(f"WT Threshold (75th pct, train): {WT_THRESHOLD:.0f}")
print(f"TC Threshold (75th pct, train): {TC_THRESHOLD:.0f}")
print(f"ET Threshold (75th pct, train): {ET_THRESHOLD:.0f}")


WT Threshold (75th pct, train): 138460
TC Threshold (75th pct, train): 58843
ET Threshold (75th pct, train): 30143


In [5]:
def _safe(row, col):
    """Return 0 if column was dropped as constant/rare in NB03."""
    return int(row[col]) if col in row.index else 0


def compute_clinical_score(df):

    """
    Compute a rule-based clinical severity score using
    quantitative MRI features and structured report features.
    """

    scores = []

    for _, row in df.iterrows():

        score = 0

        # -----------------------------------------
        # MRI Quantitative Features
        # -----------------------------------------

        if row["wt_volume"] >= WT_THRESHOLD:
            score += 2

        if row["et_volume"] >= ET_THRESHOLD:
            score += 2

        if row["tc_volume"] >= TC_THRESHOLD:
            score += 1

        # -----------------------------------------
        # Structured Report Features
        # -----------------------------------------

        if row["lobe_count"] >= 3:
            score += 1

        if row["bilateral"] == 1:
            score += 1

        scores.append(score)

    return scores


def risk_label(score):
    if score <= 1:  return 0
    if score <= 3:  return 1
    return 2


In [6]:
for split_name, data in [("train", train), ("validation", validation), ("test", test)]:
    data["clinical"]["severity_score"] = compute_clinical_score(data["clinical"])
    data["clinical"]["risk_label"]     = data["clinical"]["severity_score"].apply(risk_label)

for name, data in {"Train": train, "Validation": validation, "Test": test}.items():
    vc = data["clinical"]["risk_label"].value_counts().sort_index()
    print(f"{name}: Low={vc.get(0,0)}  Medium={vc.get(1,0)}  High={vc.get(2,0)}")


Train: Low=154  Medium=60  High=43
Validation: Low=31  Medium=9  High=16
Test: Low=34  Medium=15  High=7


## 4. Scaling Clinical Features

`StandardScaler` is **fit on train only** and applied to all splits —
matching the standard scikit-learn practice that prevents leakage of
validation/test statistics into preprocessing.


In [7]:
CLINICAL_COLUMNS = [
    c for c in [
        "wt_volume", "tc_volume", "et_volume",
        "frontal", "temporal", "parietal", "occipital",
        "left", "right", "bilateral",
        "word_count", "sentence_count", "lobe_count",
        "edema", "necrosis", "ventricle", "compression",
    ]
    if c in train["clinical"].columns   # robust to NB03 feature drops
]

print("Clinical columns used:", CLINICAL_COLUMNS)

scaler = StandardScaler()
train["clinical"][CLINICAL_COLUMNS]      = scaler.fit_transform(train["clinical"][CLINICAL_COLUMNS])
validation["clinical"][CLINICAL_COLUMNS] = scaler.transform(validation["clinical"][CLINICAL_COLUMNS])
test["clinical"][CLINICAL_COLUMNS]       = scaler.transform(test["clinical"][CLINICAL_COLUMNS])

# Save scaler so inference on new patients uses the exact same normalization
joblib.dump(scaler, MODEL_DIR / "clinical_scaler.pkl")
print("Scaler saved →", MODEL_DIR / "clinical_scaler.pkl")


Clinical columns used: ['wt_volume', 'tc_volume', 'et_volume', 'frontal', 'temporal', 'parietal', 'occipital', 'left', 'right', 'bilateral', 'word_count', 'sentence_count', 'lobe_count']
Scaler saved → /kaggle/working/models/fusion/clinical_scaler.pkl


In [8]:
# Save severity thresholds

thresholds = {

    "WT_THRESHOLD": float(WT_THRESHOLD),

    "TC_THRESHOLD": float(TC_THRESHOLD),

    "ET_THRESHOLD": float(ET_THRESHOLD),

}

with open(

    MODEL_DIR / "severity_thresholds.json",

    "w",

) as file:

    json.dump(

        thresholds,

        file,

        indent=4,

    )

print("Severity thresholds saved.")

Severity thresholds saved.


## 5. Dataset & DataLoader

In [9]:
class FusionDataset(Dataset):

    def __init__(self, image_features, text_features, clinical_features, labels):
        self.image    = image_features.astype(np.float32)
        self.text     = text_features.astype(np.float32)
        self.clinical = clinical_features.astype(np.float32)
        self.labels   = labels.astype(np.int64)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return {
            "image":    torch.tensor(self.image[idx]),
            "text":     torch.tensor(self.text[idx]),
            "clinical": torch.tensor(self.clinical[idx]),
            "label":    torch.tensor(self.labels[idx]),
        }


BATCH_SIZE     = 32
CLINICAL_DIM   = len(CLINICAL_COLUMNS)

def make_dataset(data):
    return FusionDataset(
        data["image"],
        data["text"],
        data["clinical"][CLINICAL_COLUMNS].values,
        data["clinical"]["risk_label"].values,
    )

train_dataset      = make_dataset(train)
validation_dataset = make_dataset(validation)
test_dataset       = make_dataset(test)

train_loader      = DataLoader(train_dataset,      batch_size=BATCH_SIZE, shuffle=True)
validation_loader = DataLoader(validation_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader       = DataLoader(test_dataset,       batch_size=BATCH_SIZE, shuffle=False)

batch = next(iter(train_loader))
print("image   :", batch["image"].shape)
print("text    :", batch["text"].shape)
print("clinical:", batch["clinical"].shape)
print("label   :", batch["label"].shape)
print("clinical_dim:", CLINICAL_DIM)


image   : torch.Size([32, 256])
text    : torch.Size([32, 768])
clinical: torch.Size([32, 13])
label   : torch.Size([32])
clinical_dim: 13


## 6. Model Architecture

```
Image  (256) ──┐
               ├──► FusionEncoder ──► Unified Repr (256) ──┐
Text   (768) ──┘                                            ├──► DecisionHead ──► Risk (3)
                                     Clinical (N)  ─────────┘
```


In [10]:
class ImageProjection(nn.Module):
    def __init__(self):
        super().__init__()
        self.projection = nn.Sequential(
            nn.Linear(256, 128),
            nn.LayerNorm(128),
            nn.GELU(),
            nn.Dropout(0.20),
        )
    def forward(self, x):
        return self.projection(x)


class TextProjection(nn.Module):
    def __init__(self):
        super().__init__()
        self.projection = nn.Sequential(
            nn.Linear(768, 512), nn.GELU(), nn.Dropout(0.30),
            nn.Linear(512, 256), nn.GELU(), nn.Dropout(0.20),
            nn.Linear(256, 128), nn.LayerNorm(128),
        )
    def forward(self, x):
        return self.projection(x)


class FusionBlock(nn.Module):
    def __init__(self):
        super().__init__()
        self.fusion = nn.Sequential(
            nn.Linear(256, 256), nn.GELU(), nn.Dropout(0.30),
            nn.Linear(256, 256), nn.LayerNorm(256),
        )
    def forward(self, image, text):
        return self.fusion(torch.cat([image, text], dim=1))


class FusionEncoder(nn.Module):
    """
    Projects image (256) + text (768) into a shared 256-d latent space.
    Architecture is identical to NB02 so checkpoints are compatible.
    """
    def __init__(self):
        super().__init__()
        self.image_projection = ImageProjection()
        self.text_projection  = TextProjection()
        self.fusion_block     = FusionBlock()

    def forward(self, image, text):
        return self.fusion_block(
            self.image_projection(image),
            self.text_projection(text),
        )


In [11]:
class DecisionHead(nn.Module):
    """
    Takes the 256-d fusion representation + clinical features → 3-class risk logits.
    """
    def __init__(self, fusion_dim=256, clinical_dim=13, hidden_dim=128, num_classes=3):
        super().__init__()
        self.classifier = nn.Sequential(
            nn.Linear(fusion_dim + clinical_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.30),
            nn.Linear(hidden_dim, 64),
            nn.ReLU(),
            nn.Dropout(0.20),
            nn.Linear(64, num_classes),
        )
    def forward(self, fusion_repr, clinical):
        return self.classifier(torch.cat([fusion_repr, clinical], dim=1))


In [12]:
class ClinicalDecisionModel(nn.Module):
    """
    Full end-to-end model:
        image + text → FusionEncoder → unified_repr
        unified_repr + clinical → DecisionHead → risk logits

    The unified_repr (256-d) is also exposed as a side output for
    downstream retrieval / clustering / explainability.
    """
    def __init__(self, clinical_dim):
        super().__init__()
        self.fusion_encoder = FusionEncoder()
        self.decision_head  = DecisionHead(
            fusion_dim   = 256,
            clinical_dim = clinical_dim,
            hidden_dim   = 128,
            num_classes  = 3,
        )

    def forward(self, image, text, clinical):
        unified_repr = self.fusion_encoder(image, text)
        logits       = self.decision_head(unified_repr, clinical)
        return logits, unified_repr   # both returned — repr for downstream use


model = ClinicalDecisionModel(clinical_dim=CLINICAL_DIM).to(DEVICE)
print(model)
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\nTrainable parameters: {total_params:,}")


ClinicalDecisionModel(
  (fusion_encoder): FusionEncoder(
    (image_projection): ImageProjection(
      (projection): Sequential(
        (0): Linear(in_features=256, out_features=128, bias=True)
        (1): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
        (2): GELU(approximate='none')
        (3): Dropout(p=0.2, inplace=False)
      )
    )
    (text_projection): TextProjection(
      (projection): Sequential(
        (0): Linear(in_features=768, out_features=512, bias=True)
        (1): GELU(approximate='none')
        (2): Dropout(p=0.3, inplace=False)
        (3): Linear(in_features=512, out_features=256, bias=True)
        (4): GELU(approximate='none')
        (5): Dropout(p=0.2, inplace=False)
        (6): Linear(in_features=256, out_features=128, bias=True)
        (7): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
      )
    )
    (fusion_block): FusionBlock(
      (fusion): Sequential(
        (0): Linear(in_features=256, out_features=256, bias=True)
  

## 7. Training

In [13]:
NUM_EPOCHS    = 50
LEARNING_RATE = 5e-4
WEIGHT_DECAY  = 1e-5
PATIENCE      = 10

labels = train["clinical"]["risk_label"].values

class_counts = np.bincount(labels)

class_weights = len(labels) / (
    len(class_counts) * class_counts
)

class_weights = torch.tensor(
    class_weights,
    dtype=torch.float32,
).to(DEVICE)

criterion = nn.CrossEntropyLoss(
    weight=class_weights
)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=NUM_EPOCHS,
)


def run_epoch(loader, training=True):
    model.train() if training else model.eval()
    total_loss, all_preds, all_labels = 0.0, [], []

    ctx = torch.enable_grad() if training else torch.no_grad()
    with ctx:
        for batch in loader:
            image    = batch["image"].to(DEVICE)
            text     = batch["text"].to(DEVICE)
            clinical = batch["clinical"].to(DEVICE)
            labels   = batch["label"].to(DEVICE)

            logits, _ = model(image, text, clinical)
            loss      = criterion(logits, labels)

            if training:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

            total_loss += loss.item() * len(labels)
            all_preds.extend(logits.argmax(dim=1).cpu().tolist())
            all_labels.extend(labels.cpu().tolist())

    n    = len(all_labels)
    acc  = accuracy_score(all_labels, all_preds)
    return total_loss / n, acc


best_val_acc             = 0.0
epochs_without_improvement = 0
history                  = []

print(f"{'Epoch':>6}  {'Train Loss':>10}  {'Train Acc':>9}  {'Val Loss':>8}  {'Val Acc':>7}")
print("-" * 55)

for epoch in range(1, NUM_EPOCHS + 1):

    train_loss, train_acc = run_epoch(train_loader,      training=True)
    val_loss,   val_acc   = run_epoch(validation_loader, training=False)

    scheduler.step()

    history.append({
        "epoch": epoch, "train_loss": train_loss, "train_acc": train_acc,
        "val_loss": val_loss, "val_acc": val_acc,
    })

    print(f"{epoch:>6}  {train_loss:>10.4f}  {train_acc:>9.4f}  {val_loss:>8.4f}  {val_acc:>7.4f}")

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        epochs_without_improvement = 0
        torch.save({
            "epoch":          epoch,
            "model_state":    model.state_dict(),
            "optimizer_state": optimizer.state_dict(),
            "val_acc":        best_val_acc,
            "clinical_dim":   CLINICAL_DIM,
            "clinical_cols":  CLINICAL_COLUMNS,
        }, MODEL_DIR / "best_decision_model.pth")
    else:
        epochs_without_improvement += 1
        if epochs_without_improvement >= PATIENCE:
            print(f"\nEarly stopping at epoch {epoch} — no improvement for {PATIENCE} epochs.")
            break

    # unconditional per-epoch save (crash recovery)
    torch.save({
        "epoch":          epoch,
        "model_state":    model.state_dict(),
        "optimizer_state": optimizer.state_dict(),
        "val_acc":        val_acc,
        "clinical_dim":   CLINICAL_DIM,
        "clinical_cols":  CLINICAL_COLUMNS,
    }, MODEL_DIR / "last_decision_model.pth")

print(f"\nBest Validation Accuracy: {best_val_acc:.4f}")


 Epoch  Train Loss  Train Acc  Val Loss  Val Acc
-------------------------------------------------------
     1      1.1091     0.3152    1.1001   0.1429
     2      1.0897     0.3774    1.0970   0.5536
     3      1.1058     0.5720    1.1352   0.5536
     4      1.0883     0.5798    1.0477   0.8036
     5      1.0653     0.5058    1.0341   0.3036
     6      1.0777     0.3268    1.0125   0.7857
     7      1.0404     0.4942    0.9411   0.7679
     8      0.9978     0.5759    0.9031   0.7857
     9      0.8948     0.6965    0.7078   0.7679
    10      0.7979     0.7121    0.6149   0.8214
    11      0.7386     0.7198    0.6664   0.6786
    12      0.7121     0.6887    0.5140   0.8571
    13      0.8096     0.7121    0.6535   0.6964
    14      0.7285     0.7393    0.5464   0.8214
    15      0.6727     0.7549    0.5477   0.8750
    16      0.6879     0.7743    0.5193   0.8393
    17      0.6762     0.7743    0.5010   0.8571
    18      0.7274     0.7782    0.5906   0.7500
    19      0

## 8. Test Evaluation

In [14]:
# Load best checkpoint before evaluating on test
checkpoint = torch.load(MODEL_DIR / "best_decision_model.pth", map_location=DEVICE)
model.load_state_dict(checkpoint["model_state"])
model.eval()

all_preds, all_labels = [], []
with torch.no_grad():
    for batch in test_loader:
        logits, _ = model(
            batch["image"].to(DEVICE),
            batch["text"].to(DEVICE),
            batch["clinical"].to(DEVICE),
        )
        all_preds.extend(logits.argmax(dim=1).cpu().tolist())
        all_labels.extend(batch["label"].tolist())

print("=" * 55)
print("Test Set Evaluation")
print("=" * 55)
print(classification_report(
    all_labels, all_preds,
    target_names=["Low Risk", "Medium Risk", "High Risk"],
    digits=4,
))


Test Set Evaluation
              precision    recall  f1-score   support

    Low Risk     0.9143    0.9412    0.9275        34
 Medium Risk     0.8000    0.5333    0.6400        15
   High Risk     0.5455    0.8571    0.6667         7

    accuracy                         0.8214        56
   macro avg     0.7532    0.7772    0.7447        56
weighted avg     0.8376    0.8214    0.8179        56



## 9. Unified Representation Extraction

Extract the 256-d fusion representation for **every patient** across all
three splits. These vectors are saved as `.npy` files alongside a
`patient_id` metadata CSV — ready for:

- Similar patient retrieval
- Clustering analysis
- Explainability (SHAP / attention)
- Any downstream task that needs a compact multimodal embedding


In [15]:
def extract_representations(loader):

    model.eval()

    representations = []

    predictions = []

    confidences = []

    labels = []

    with torch.no_grad():

        for batch in loader:

            image = batch["image"].to(DEVICE)

            text = batch["text"].to(DEVICE)

            clinical = batch["clinical"].to(DEVICE)

            target = batch["label"].to(DEVICE)

            logits, unified_repr = model(

                image,

                text,

                clinical,

            )

            probabilities = torch.softmax(

                logits,

                dim=1,

            )

            confidence = probabilities.max(

                dim=1

            ).values

            prediction = probabilities.argmax(

                dim=1

            )

            representations.extend(

                unified_repr.cpu().numpy()

            )

            predictions.extend(

                prediction.cpu().numpy()

            )

            confidences.extend(

                confidence.cpu().numpy()

            )

            labels.extend(

                target.cpu().numpy()

            )

    return (

        np.array(representations),

        np.array(predictions),

        np.array(confidences),

        np.array(labels),

    )

SPLIT_DATASETS = {

    "train": (

        FusionDataset(

            train["image"],

            train["text"],

            train["clinical"][CLINICAL_COLUMNS].values,

            train["clinical"]["risk_label"].values,

        ),

        train["clinical"],

    ),

    "validation": (

        FusionDataset(

            validation["image"],

            validation["text"],

            validation["clinical"][CLINICAL_COLUMNS].values,

            validation["clinical"]["risk_label"].values,

        ),

        validation["clinical"],

    ),

    "test": (

        FusionDataset(

            test["image"],

            test["text"],

            test["clinical"][CLINICAL_COLUMNS].values,

            test["clinical"]["risk_label"].values,

        ),

        test["clinical"],

    ),

}


for split_name, (dataset, clinical_df) in SPLIT_DATASETS.items():

    loader = DataLoader(

        dataset,

        batch_size=BATCH_SIZE,

        shuffle=False,

    )

    representations, predictions, confidences, labels = extract_representations(

        loader

    )

    metadata = clinical_df[

        [

            "patient_id",

            "risk_label",

            "severity_score",

        ]

    ].copy().reset_index(drop=True)

    metadata["predicted_risk"] = predictions

    metadata["confidence"] = confidences

    np.save(

        REPR_DIR /

        f"{split_name}_unified_repr.npy",

        representations,

    )

    metadata.to_csv(

        REPR_DIR /

        f"{split_name}_repr_metadata.csv",

        index=False,

    )

    print(

        f"{split_name:<12}",

        representations.shape,

    )

train        (257, 256)
validation   (56, 256)
test         (56, 256)


## 10. Save Training History & Final Summary

In [16]:
history_df = pd.DataFrame(history)
history_df.to_csv(MODEL_DIR / "training_history.csv", index=False)

print("=" * 60)
print("ClinicalDecisionModel — Training Complete")
print("=" * 60)
print(f"  Best Val Accuracy  : {best_val_acc:.4f}")
print(f"  Clinical dim used  : {CLINICAL_DIM}")
print(f"  Clinical columns   : {CLINICAL_COLUMNS}")
print()
print("Saved artifacts:")
print(f"  {MODEL_DIR / 'best_decision_model.pth'}")
print(f"  {MODEL_DIR / 'last_decision_model.pth'}")
print(f"  {MODEL_DIR / 'clinical_scaler.pkl'}")
print(f"  {MODEL_DIR / 'training_history.csv'}")
print(f"  {REPR_DIR}  (unified_repr + metadata per split)")


ClinicalDecisionModel — Training Complete
  Best Val Accuracy  : 0.8750
  Clinical dim used  : 13
  Clinical columns   : ['wt_volume', 'tc_volume', 'et_volume', 'frontal', 'temporal', 'parietal', 'occipital', 'left', 'right', 'bilateral', 'word_count', 'sentence_count', 'lobe_count']

Saved artifacts:
  /kaggle/working/models/fusion/best_decision_model.pth
  /kaggle/working/models/fusion/last_decision_model.pth
  /kaggle/working/models/fusion/clinical_scaler.pkl
  /kaggle/working/models/fusion/training_history.csv
  /kaggle/working/reports/fusion/representations  (unified_repr + metadata per split)


In [17]:
np.save(
    MODEL_DIR / "class_weights.npy",
    class_weights.cpu().numpy()
)